# 包裹运输成本问题求解设计

## 一、模型选择与核心设定
### 1. 模型类型
采用 **k-近邻回归（KNeighborsRegressor）**，适用于连续型目标变量（运输成本y）的预测，核心逻辑是通过计算新样本与训练样本的距离，选取近邻样本的运输成本均值，捕捉包裹属性（体积x1、重量x2）与运输成本的关联关系。

### 2. 变量定义
- **目标变量**：包裹运输成本y；
- **特征变量**：包裹体积x1、重量x2两个数值型特征。

### 3. 模型具体形式
- **特征预处理**：x1和x2数值量级一致（均为1.x-4.x），无需标准化，直接计算样本间距离；
- **核心参数**：`n_neighbors=3`（k值），数据集共10个样本，选择k=3（奇数避免投票平局，适配小样本规模）；
- **距离度量**：欧式距离（适用于低维连续特征，精准计算样本间相似性）；
- **预测规则**：对k个最近邻样本的运输成本取算术平均值，作为新包裹的运输成本预测值。

## 二、模型构建与评估流程
### 1. 数据集构建
手动构建训练数据集，包含特征矩阵（x1、x2）与目标变量向量（运输成本y），共10个样本。

### 2. 新样本预测流程
1. 计算新包裹（x1=3.8、x2=3.2）与所有训练样本的欧式距离；
2. 选取距离最小的k=3个近邻样本；
3. 取3个近邻样本的运输成本均值，作为新包裹的运输成本预测值。

### 3. 模型效果评估
- 评估方法：因样本量极小（仅10个），采用**留一交叉验证（LOOCV）**，最大化利用有限样本评估模型泛化能力；
- 评估指标：计算LOOCV的RMSE（均方根误差）、MAE（平均绝对误差），量化模型拟合效果。

## 三、技术路线
1. 手动构建包裹运输成本训练数据集（特征矩阵+目标变量向量）；
2. 定义待预测新包裹特征：体积3.8、重量3.2；
3. 计算新样本与训练样本的欧式距离，筛选k=3个最近邻；
4. 取近邻运输成本均值，得到新包裹的运输成本预测值；
5. 采用留一交叉验证（LOOCV）评估模型整体效果；
6. 输出预测结果与模型评估指标（RMSE、MAE）。

In [4]:
# 包裹运输成本预测 - k-近邻回归模型
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error
# ===================== 1. 构建训练数据集 =====================
# 特征矩阵X：每一行对应(x1, x2)，即体积、重量
X = np.array([
    [2.0, 3.5],
    [3.1, 2.8],
    [4.2, 1.9],
    [1.8, 4.3],
    [3.9, 3.0],
    [2.7, 2.5],
    [4.1, 1.6],
    [2.5, 3.9],
    [3.4, 3.4],
    [3.7, 2.3]
])
# 目标变量y：运输成本
y = np.array([7.8, 6.2, 5.5, 8.5, 7.1, 6.4, 5.1, 8.0, 6.8, 6.0])

# 打印数据集信息
print("="*60)
print("包裹运输成本训练数据集")
print("="*60)
print("特征（体积x1, 重量x2）| 运输成本y")
for i in range(len(X)):
    print(f"({X[i][0]:.1f}, {X[i][1]:.1f}) | {y[i]:.1f}")
print("="*60)

# ===================== 2. 定义待预测新包裹 =====================
new_package = np.array([[3.8, 3.2]])  # 体积3.8、重量3.2，注意二维数组格式
print(f"\n待预测包裹：体积={new_package[0][0]}, 重量={new_package[0][1]}")

# ===================== 3. 构建kNN回归模型并预测 =====================
k = 3  # 选择3个最近邻
knn_reg = KNeighborsRegressor(n_neighbors=k, metric='euclidean')
knn_reg.fit(X, y)  # 训练模型

# 预测新包裹的运输成本
y_pred = knn_reg.predict(new_package)
print(f"\n选择k={k}个最近邻，预测运输成本：{y_pred[0]:.2f}")

# 提取k个最近邻的信息（辅助解读）
distances, neighbors_indices = knn_reg.kneighbors(new_package)
neighbors_X = X[neighbors_indices[0]]
neighbors_y = y[neighbors_indices[0]]
print(f"\n{k}个最近邻样本：")
for i, idx in enumerate(neighbors_indices[0]):
    print(f"近邻{i+1}：(x1={X[idx][0]:.1f}, x2={X[idx][1]:.1f}), 运输成本={y[idx]:.1f}, 距离={distances[0][i]:.3f}")
print(f"近邻运输成本均值：{np.mean(neighbors_y):.2f}（与预测值一致）")

# ===================== 4. 留一交叉验证评估模型效果 =====================
print("\n" + "="*60)
print("模型效果评估（留一交叉验证）")
print("="*60)
loo = LeaveOneOut()  # 留一交叉验证（适配小样本）
# 计算LOOCV的RMSE
cv_rmse = np.sqrt(-cross_val_score(knn_reg, X, y, cv=loo, scoring='neg_mean_squared_error').mean())
# 计算LOOCV的MAE
cv_mae = -cross_val_score(knn_reg, X, y, cv=loo, scoring='neg_mean_absolute_error').mean()
print(f"留一交叉验证 RMSE：{cv_rmse:.3f}")
print(f"留一交叉验证 MAE：{cv_mae:.3f}")

# ===================== 5. 模型拟合效果补充评估 =====================
# 训练集上的预测值与真实值对比
y_train_pred = knn_reg.predict(X)
train_rmse = np.sqrt(mean_squared_error(y, y_train_pred))
train_mae = mean_absolute_error(y, y_train_pred)
print(f"\n训练集 RMSE：{train_rmse:.3f}")
print(f"训练集 MAE：{train_mae:.3f}")
print("="*60)

包裹运输成本训练数据集
特征（体积x1, 重量x2）| 运输成本y
(2.0, 3.5) | 7.8
(3.1, 2.8) | 6.2
(4.2, 1.9) | 5.5
(1.8, 4.3) | 8.5
(3.9, 3.0) | 7.1
(2.7, 2.5) | 6.4
(4.1, 1.6) | 5.1
(2.5, 3.9) | 8.0
(3.4, 3.4) | 6.8
(3.7, 2.3) | 6.0

待预测包裹：体积=3.8, 重量=3.2

选择k=3个最近邻，预测运输成本：6.70

3个最近邻样本：
近邻1：(x1=3.9, x2=3.0), 运输成本=7.1, 距离=0.224
近邻2：(x1=3.4, x2=3.4), 运输成本=6.8, 距离=0.447
近邻3：(x1=3.1, x2=2.8), 运输成本=6.2, 距离=0.806
近邻运输成本均值：6.70（与预测值一致）

模型效果评估（留一交叉验证）
留一交叉验证 RMSE：0.581
留一交叉验证 MAE：0.470

训练集 RMSE：0.288
训练集 MAE：0.250


### 一、具体模型信息
最终构建的**k-近邻回归模型**配置如下：
1. **模型类型**：k-近邻回归（KNeighborsRegressor），适用于小样本下连续型目标变量（运输成本）的预测；
2. **特征变量**：包裹体积x1、重量x2（无特征预处理，直接使用原始数值，因特征量级一致）；
3. **核心参数**：`n_neighbors=3`（选择3个最近邻样本），距离度量采用欧式距离；
4. **预测规则**：对3个最近邻样本的运输成本取算术平均值，作为新包裹的运输成本预测值。

### 二、模型效果评估
#### 1. 留一交叉验证（适配小样本评估）
| 评估指标       | 数值       | 说明                     |
|----------------|------------|--------------------------|
| RMSE（均方根误差） | 0.581      | 模型整体预测的平均偏差约0.581 |
| MAE（平均绝对误差） | 0.470      | 模型平均绝对预测偏差约0.470 |

#### 2. 训练集拟合效果
| 评估指标       | 数值       | 说明                     |
|----------------|------------|--------------------------|
| RMSE（均方根误差） | 0.288      | 模型在训练集上的拟合偏差约0.288 |
| MAE（平均绝对误差） | 0.250      | 模型在训练集上的平均绝对拟合偏差约0.250 |

#### 3. 效果解读
- 训练集RMSE和MAE数值较小，说明模型能有效捕捉包裹体积、重量与运输成本的关联关系，拟合效果良好；
- 留一交叉验证指标略高于训练集，符合小样本数据的特性，无明显过拟合，模型泛化能力稳定。

### 三、运输成本预测结果
1. **待预测包裹属性**：体积3.8、重量3.2；
2. **3个最近邻样本详情**：
   - 近邻1：(x1=3.9, x2=3.0)，运输成本7.1，与新包裹的欧式距离0.224；
   - 近邻2：(x1=3.4, x2=3.4)，运输成本6.8，与新包裹的欧式距离0.447；
   - 近邻3：(x1=3.1, x2=2.8)，运输成本6.2，与新包裹的欧式距离0.806；
3. **最终预测结果**：该包裹的运输成本为 **6.70**（即3个近邻运输成本的均值：(7.1+6.8+6.2)/3 = 6.70）。